In [2]:
# Target variable: converted

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('data.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [4]:
for c in df.columns:
    filler = 'NA' if df[c].dtype == 'str' else 0
    df[c] = df[c].fillna(filler)

df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [5]:
# Question one

df.industry.mode()

0    technology
Name: industry, dtype: str

In [12]:
# Question two

numerical = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
df[numerical].corr().round(2)

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.00,0.16,0.12,0.23
number_of_courses_viewed,0.16,1.00,0.72,0.76
interaction_count,0.12,0.72,1.00,0.92
lead_score,0.23,0.76,0.92,1.00


In [9]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_full_train = df_full_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_train['converted'] 
del df_val['converted']
del df_test['converted']


In [10]:
# Question three

from sklearn.metrics import mutual_info_score

categorical = ['lead_source', 'industry', 'employment_status', 'location']

def mutual_info_converted_score(series):
    return mutual_info_score(series, df_full_train.converted)

mi = df_full_train[categorical].apply(mutual_info_converted_score)
mi.sort_values(ascending=False)

lead_source          0.034761
employment_status    0.020313
industry             0.002898
location             0.001050
dtype: float64

In [28]:
from sklearn.feature_extraction import DictVectorizer

train_dict = df_train[categorical + numerical].to_dict(orient='records')
val_dict = df_val[categorical + numerical].to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dict)
X_val = dv.transform(val_dict)

In [29]:
# Question four

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]

converted_decision = (y_pred >= 0.5)
original_accuracy = (y_val == converted_decision).mean()
original_accuracy

np.float64(0.645)

In [25]:
# Question five

features = list(df_train.columns)

for feature in features:
    subset_features = [f for f in features if f != feature]
    train_dict = df_train[subset_features].to_dict(orient='records')
    val_dict = df_val[subset_features].to_dict(orient='records')

    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(train_dict)
    X_val = dv.transform(val_dict)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]
    converted_decision = (y_pred >= 0.5)
    print(f'Accuracy without \'{feature}\': {original_accuracy - (y_val == converted_decision).mean()}\n')

Accuracy without 'lead_source': 0.0030000000000000027

Accuracy without 'industry': 0.0

Accuracy without 'employment_status': 0.0

Accuracy without 'location': 0.0

Accuracy without 'annual_income': -0.07899999999999996

Accuracy without 'number_of_courses_viewed': 0.0020000000000000018

Accuracy without 'interaction_count': 0.04400000000000004

Accuracy without 'lead_score': 0.0010000000000000009



In [30]:
# Question six

C = [0.000001, 0.00001, 0.0001, 0.001]
for reg in C:
    model = LogisticRegression(solver='liblinear', C=reg, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]

    converted_decision = (y_pred >= 0.5)
    print((y_val == converted_decision).mean().round(3))
    print()

0.598

0.598

0.613

0.645

